<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-10c-training-at-scale-ddp-zero-fsdp-and-cost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Discussion 10 (continued) - Training at scale: memory arithmetic, data parallelism, ZeRO / FSDP, and the cost of a GPU-hour

What really fills a GPU during training, how data parallelism and ZeRO/FSDP fix it (verified on CPU with real `torch.distributed` processes), and how to turn FLOPs, utilisation and spot interruptions into dollars.

> ## ⚠️ Multi-GPU needs paid hardware
> Parts 1-5 run **free on the Colab CPU** (they use several CPU processes to behave like several GPUs). **Colab only ever gives you one GPU**, so real multi-GPU training (Part 6) needs a **cloud instance with 2-8 GPUs** - for example one bought with the free-credit route from Chapter 15, or any rented multi-GPU machine. Prices and availability change; shut instances down when you finish. Part 6 is optional and nothing graded depends on it.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate.

*Original code written for this course. Facts follow the ZeRO paper (Rajbhandari et al., 2020, arXiv:1910.02054) and PyTorch's FSDP documentation; links are in the chapter.*

In [ ]:
%pip install -q numpy torch

In [ ]:
import math, os, subprocess, sys, numpy as np
np.random.seed(0)

## Part 1 - What fills the GPU: 16 bytes per parameter

Training with mixed precision and Adam keeps, for every parameter: the working weight in bf16/fp16 (2 bytes), its gradient (2 bytes), and optimizer state in fp32 - a master copy of the weight (4), Adam's first moment (4) and second moment (4) - so `2 + 2 + K` bytes with `K = 12`. That is **16 bytes per parameter before a single activation is stored**, eight times the size of the bf16 weights alone.

In [ ]:
def train_state_gb(params_b, bytes_per_param=16):
    return params_b * 1e9 * bytes_per_param / 1e9

print(f"{'model':>6} {'weights bf16':>13} {'full fine-tune states':>22}")
for n in (1, 7, 13, 70):
    print(f"{n:>5}B {2*n:>11.0f}GB {train_state_gb(n):>20.0f}GB")
print("\n(model states only - activations, buffers and fragmentation come on top)")

**Read it.** A 7B model needs about 112 GB just for its training states: more than one 80 GB GPU, before activations. That is why a 7B *full* fine-tune cannot happen on one A100-80GB, but a *LoRA* fine-tune of the same model can (Chapter 10): LoRA freezes the base weights (so no gradients or optimizer state for them) and trains a tiny adapter.

In [ ]:
def lora_train_gb(params_b, trainable_frac=0.01, base_bytes=2.0):
    base = params_b * 1e9 * base_bytes                       # frozen weights, bf16 (2) or 4-bit (~0.55 with scales)
    adapter = params_b * 1e9 * trainable_frac * 16           # adapters train with the full 16 bytes/param
    return (base + adapter) / 1e9

for label, base_b in (("LoRA, bf16 base", 2.0), ("QLoRA, 4-bit base", 0.5 + 2/128)):
    print(f"7B {label:18}: {lora_train_gb(7, 0.01, base_b):6.1f} GB of states   (full fine-tune: {train_state_gb(7):.0f} GB)")

## Part 2 - ZeRO: stop storing the same thing on every GPU

Plain data parallelism puts a full copy of the 16 bytes/param on **every** GPU. ZeRO (Rajbhandari et al.) observes that this is redundant and shards the states across the `N` data-parallel GPUs, in three stages:

| Stage | What is sharded across GPUs | Per-GPU model-state memory (`P` params, `K=12`) |
|---|---|---|
| 0 (plain DP) | nothing | `(2 + 2 + K) P = 16 P` |
| 1 | optimizer state | `(2 + 2) P + K P / N` |
| 2 | + gradients | `2 P + (2 + K) P / N` |
| 3 | + the parameters themselves | `(2 + 2 + K) P / N` |

PyTorch's FSDP implements the stage-3 idea natively (DeepSpeed implements all three). The function below reproduces the example in the paper (7.5B parameters, 64 GPUs: 120, 31.4, 16.6 and 1.9 GB).

In [ ]:
def zero_gb(params_b, n_gpus, stage, K=12):
    P = params_b
    return {0: (2 + 2 + K) * P,
            1: (2 + 2) * P + K * P / n_gpus,
            2: 2 * P + (2 + K) * P / n_gpus,
            3: (2 + 2 + K) * P / n_gpus}[stage]

print("7.5B model, 64 GPUs (the paper's example):", [round(zero_gb(7.5, 64, s), 2) for s in range(4)], "GB per GPU")
print()
print(f"{'GPUs':>5} " + "".join(f"{'stage '+str(s):>10}" for s in range(4)) + "   (7B model, GB per GPU, states only)")
for n in (1, 2, 4, 8, 16):
    print(f"{n:>5} " + "".join(f"{zero_gb(7, n, s):>10.1f}" for s in range(4)))

**Read it.** A 7B full fine-tune (112 GB of states) fits 80 GB GPUs once it is sharded: 2 GPUs at stage 2 or 3 (63 or 56 GB per GPU), or 4 GPUs at stage 1 (49 GB). How much room is left for activations - 17 GB, 24 GB or 31 GB - is what decides your batch size. The price is communication: plain data parallelism moves roughly `2P` numbers per step (an all-reduce of gradients), whereas stage 3 moves about `3P` (the paper reports 1.5x the baseline), because parameters must be gathered before use. **You trade network traffic for memory.**

## Part 3 - It is just arithmetic: verify it with real processes

The script below launches several operating-system processes that talk through `torch.distributed` (the `gloo` backend, which runs on CPU). It checks three claims against a single-process reference:

1. **Data parallel is exact:** splitting a batch across ranks and averaging the gradients with all-reduce gives the same gradient as one process seeing the whole batch.
2. **ZeRO-style sharding is exact:** keeping Adam's moments for only 1/N of the parameters on each rank, then all-gathering the updated shards, gives the same weights as ordinary Adam, with N times less optimizer state per rank.
3. **PyTorch's real FSDP2** (`fully_shard`) runs, and its parameters become sharded `DTensor`s.

In [ ]:
DDP_SRC = '"""Data parallelism and ZeRO-style sharding, by hand, on CPU (gloo backend).\nRun:  python ddp_demo.py 4      (4 processes)"""\nimport os, sys, torch, torch.nn as nn, torch.distributed as dist, torch.multiprocessing as mp\n\ndef make_model():\n    torch.manual_seed(0)                                   # every rank builds identical weights\n    return nn.Sequential(nn.Linear(16, 32), nn.Tanh(), nn.Linear(32, 4))\n\ndef flat(params):  return torch.cat([p.reshape(-1) for p in params])\ndef flat_grad(m):  return torch.cat([p.grad.reshape(-1) for p in m.parameters()])\ndef set_flat(m, v):\n    i = 0\n    for p in m.parameters():\n        n = p.numel(); p.data.copy_(v[i:i + n].view_as(p)); i += n\n\ndef worker(rank, world, port):\n    os.environ.update(MASTER_ADDR="127.0.0.1", MASTER_PORT=str(port))\n    dist.init_process_group("gloo", rank=rank, world_size=world)\n    g = torch.Generator().manual_seed(1)\n    X, y = torch.randn(32, 16, generator=g), torch.randint(0, 4, (32,), generator=g)   # one GLOBAL batch, same on every rank\n    lossf = nn.CrossEntropyLoss()\n\n    # --- 1. reference: one process sees the whole batch ---\n    ref = make_model(); lossf(ref(X), y).backward(); g_ref = flat_grad(ref)\n\n    # --- 2. data parallel: each rank sees 1/world of the batch, then gradients are averaged (all-reduce) ---\n    m = make_model()\n    xs, ys = X.chunk(world)[rank], y.chunk(world)[rank]\n    lossf(m(xs), ys).backward()\n    g_local = flat_grad(m).clone()\n    gsum = g_local.clone(); dist.all_reduce(gsum); g_dp = gsum / world\n    if rank == 0:\n        print(f"[DDP]  ranks={world}  local batch={len(xs)}  max|averaged grad - single-process grad| = {(g_dp - g_ref).abs().max():.2e}")\n\n    # --- 3. ZeRO-1/2 by hand: shard Adam state (and gradient) across ranks ---\n    n = g_dp.numel(); pad = (-n) % world; shard = (n + pad) // world\n    params = F = torch.cat([flat([p for p in make_model().parameters()]), torch.zeros(pad)])          # full flat params (same everywhere)\n    lo, hi = rank * shard, (rank + 1) * shard\n    my_p = params[lo:hi].clone()                                                                      # this rank owns one shard\n    gfull = torch.cat([g_local, torch.zeros(pad)]); dist.all_reduce(gfull); gfull /= world            # (a real stack uses reduce-scatter: only the shard arrives)\n    my_g = gfull[lo:hi]\n    m1, m2 = torch.zeros(shard), torch.zeros(shard)                                                   # Adam moments exist ONLY for this shard\n    lr, b1, b2, eps = 1e-2, 0.9, 0.999, 1e-8\n    m1 = b1 * m1 + (1 - b1) * my_g; m2 = b2 * m2 + (1 - b2) * my_g ** 2\n    my_p -= lr * (m1 / (1 - b1)) / ((m2 / (1 - b2)).sqrt() + eps)\n    parts = [torch.zeros(shard) for _ in range(world)]; dist.all_gather(parts, my_p)                  # all-gather the updated shards\n    new_zero = torch.cat(parts)[:n]\n\n    # reference Adam step with full state on every rank\n    mm = make_model(); opt = torch.optim.Adam(mm.parameters(), lr=lr)\n    lossf(mm(X), y).backward(); opt.step(); new_ref = flat(list(mm.parameters())).detach()\n    state_elems = sum(v.numel() for s in opt.state.values() for k, v in s.items() if torch.is_tensor(v) and v.dim() > 0)\n    if rank == 0:\n        print(f"[ZeRO] params={n}  Adam state per rank: {m1.numel() + m2.numel()} numbers (vs {state_elems} unsharded) -> {state_elems / (m1.numel() + m2.numel()):.1f}x less")\n        print(f"[ZeRO] max|sharded-update params - ordinary Adam params| = {(new_zero - new_ref).abs().max():.2e}")\n\n    # --- 4. the real thing: PyTorch FSDP2 (fully_shard) on CPU ---\n    try:\n        from torch.distributed.fsdp import fully_shard\n        fm = make_model(); fully_shard(fm)\n        out = lossf(fm(xs), ys); out.backward()\n        if rank == 0:\n            print(f"[FSDP2] fully_shard ran on CPU/gloo: loss={out.item():.4f}; param type={type(next(fm.parameters())).__name__}; local shard shape={tuple(next(fm.parameters()).to_local().shape)}")\n    except Exception as e:\n        if rank == 0:\n            print(f"[FSDP2] not runnable on this build/backend: {type(e).__name__}: {str(e)[:120]}")\n    dist.destroy_process_group()\n\nif __name__ == "__main__":\n    world = int(sys.argv[1]) if len(sys.argv) > 1 else 2\n    mp.spawn(worker, args=(world, 29511), nprocs=world, join=True)\n'
open('ddp_demo.py', 'w', encoding='utf8').write(DDP_SRC)
print('wrote ddp_demo.py,', len(DDP_SRC.splitlines()), 'lines')

In [ ]:
r = subprocess.run([sys.executable, 'ddp_demo.py', '4'], capture_output=True, text=True, timeout=300)
print('\n'.join(l for l in r.stdout.splitlines()))
if r.returncode: print(r.stderr[-1500:])

Differences around `1e-8` are floating-point rounding, not algorithmic error. The same ideas, at scale, are what `DistributedDataParallel`, DeepSpeed ZeRO and FSDP do - with overlap of communication and computation, bucketing, and reduce-scatter instead of the all-reduce we used for clarity.

## Part 4 - How fast, and how efficiently? FLOPs and MFU

For a dense transformer, training costs about `6 N D` floating-point operations (`N` parameters, `D` training tokens: 2 for the forward pass, 4 for the backward). **Model FLOPs utilisation (MFU)** is the fraction of the GPU's peak you actually achieve: `MFU = 6 N (tokens/s) / (GPUs x peak FLOP/s)`. Good large-scale runs report roughly 30-55%; small, badly tuned runs often sit below 20%. Peak figures are datasheet dense bf16/fp16 tensor-core numbers.

In [ ]:
PEAK_TFLOPS = {"L4": 121, "A100": 312, "H100": 989}        # dense bf16/fp16 tensor-core peak (datasheet)

def gpu_hours(params_b, tokens_b, gpu, mfu):
    flops = 6 * params_b * 1e9 * tokens_b * 1e9
    return flops / (PEAK_TFLOPS[gpu] * 1e12 * mfu) / 3600

print("Fine-tune 7B on 1B tokens:")
print(f"{'GPU':>5} {'MFU':>5} {'GPU-hours':>10}")
for gpu in PEAK_TFLOPS:
    for mfu in (0.2, 0.4):
        print(f"{gpu:>5} {mfu:>5.0%} {gpu_hours(7, 1, gpu, mfu):>10.0f}")

**Read it.** Doubling MFU halves the bill without buying anything. Typical wins, in the order to try them: larger micro-batches until memory is full, `bf16`, fused/flash attention kernels, `torch.compile`, packing sequences to avoid padding, and overlapping communication with compute. **Activation checkpointing** (recompute activations in the backward pass instead of storing them) saves memory at the price of roughly one extra forward pass, about +33% compute; use it when memory, not speed, is what stops you.

## Part 5 - Spot instances and checkpoint intervals

Spot / preemptible GPUs are much cheaper but can vanish. If a job saves a checkpoint every `T` seconds, each checkpoint costs `C` seconds, and interruptions arrive on average every `M` seconds, the classic Young/Daly rule says the best interval is about `sqrt(2 C M)`. Too frequent and you waste time saving; too rare and you lose a lot of work at each interruption. We test the rule with a small Monte Carlo simulation.

In [ ]:
def wall_time(work_h, interval_h, ckpt_h, mtbf_h, trials=400, restart_h=0.1, seed=0):
    rng = np.random.default_rng(seed); out = []
    for _ in range(trials):
        done = t = 0.0; next_fail = rng.exponential(mtbf_h)
        while done < work_h:
            seg = min(interval_h, work_h - done)
            seg_time = seg + (ckpt_h if done + seg < work_h else 0)
            if t + seg_time <= next_fail:
                t += seg_time; done += seg
            else:                                      # interrupted: lose this segment, restart from the last checkpoint
                t = next_fail + restart_h; next_fail = t + rng.exponential(mtbf_h)
        out.append(t)
    return float(np.mean(out))

work, C, M = 40.0, 2/60, 4.0                             # 40 h job, 2-minute checkpoint, interruption every 4 h on average
opt = math.sqrt(2 * C * M)
print(f"Young/Daly interval = sqrt(2*C*M) = {opt:.2f} h ({opt*60:.0f} min)")
print(f"{'interval':>10} {'mean wall-clock (h)':>20}")
for iv in (0.05, 0.1, opt, 0.75, 1.5, 3.0):
    print(f"{iv*60:>7.0f} min {wall_time(work, iv, C, M):>20.1f}" + ("   <- Young/Daly" if iv == opt else ""))

A spot GPU at 30-60% of the on-demand price is a bargain only if the extra wall-clock time (and your engineering time) is smaller than the discount - and only if your training loop can resume from a checkpoint, including the optimizer state and the data-loader position.

## Part 6 - OPTIONAL, PAID MULTI-GPU: run it for real

**Requires a machine with 2 or more NVIDIA GPUs** (a rented cloud instance; Colab offers one GPU only). Skipped automatically if fewer than 2 GPUs are visible. Launch with `torchrun`, one process per GPU:

```bash
torchrun --standalone --nproc_per_node=<number of GPUs> fsdp_train.py
```

The script below is a small FSDP2 training loop that prints peak GPU memory per rank; run it once with plain DDP wrapping and once with `fully_shard`, and compare the memory column. Nothing here is graded.

In [ ]:
import torch
n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
print("GPUs visible:", n_gpu)
FSDP_SRC = """
import os, sys, torch, torch.nn as nn, torch.distributed as dist
from torch.distributed.fsdp import fully_shard
mode = sys.argv[1] if len(sys.argv) > 1 else "fsdp"          # "ddp" or "fsdp"
dist.init_process_group("nccl"); rank = dist.get_rank(); torch.cuda.set_device(rank)
dim = 4096
model = nn.Sequential(*[nn.Sequential(nn.Linear(dim, dim), nn.GELU()) for _ in range(16)]).cuda().bfloat16()
if mode == "ddp": model = nn.parallel.DistributedDataParallel(model, device_ids=[rank])
else:
    for blk in model: fully_shard(blk)
    fully_shard(model)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
x = torch.randn(8, dim, device="cuda", dtype=torch.bfloat16)
for step in range(10):
    loss = model(x).float().pow(2).mean(); loss.backward(); opt.step(); opt.zero_grad()
if rank == 0:
    print(f"mode={mode} world={dist.get_world_size()} peak GPU memory on rank 0 = {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
dist.destroy_process_group()
"""
open("fsdp_train.py", "w").write(FSDP_SRC)
if n_gpu >= 2:
    for mode in ("ddp", "fsdp"):
        r = subprocess.run(["torchrun", "--standalone", f"--nproc_per_node={n_gpu}", "fsdp_train.py", mode], capture_output=True, text=True)
        print(r.stdout.strip() or r.stderr[-800:])
else:
    print("Fewer than 2 GPUs - skipping the live run. The script was written to fsdp_train.py so you can copy it to a multi-GPU machine.")

## What to hand in if you ran Part 6

GPU model and count, interconnect if known (NVLink or PCIe), PyTorch version, the two printed memory lines (DDP versus FSDP), and one sentence on what you would shard next if the model were 4x larger.